# 🛒 ShopEase — SQL Data Analysis
> **Celebal Technologies Summer Internship — Week 2**  
> Analyst: Himanshi | Database: SQLite | Language: Python + SQL

---
## 📋 Business Context
ShopEase is a mid-sized e-commerce company selling **Electronics**, **Clothing**, and **Home** products across India.  
We analyze customer behavior, product performance, and sales trends.

## 🗃️ Schema
```
customers ──(1:N)──▶ orders ──(1:N)──▶ order_items ◀──(1:N)── products
```

In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

conn = sqlite3.connect('shopease.db')

def run(sql):
    return pd.read_sql_query(sql, conn)

print('✅ Connected to shopease.db')

✅ Connected to shopease.db


---
## 📦 Setup — Create Tables & Load Data

In [2]:
cur = conn.cursor()
cur.executescript("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    first_name  VARCHAR(50)  NOT NULL,
    last_name   VARCHAR(50)  NOT NULL,
    email       VARCHAR(100) UNIQUE NOT NULL,
    city        VARCHAR(50)  NOT NULL,
    state       VARCHAR(50)  NOT NULL,
    join_date   DATE         NOT NULL,
    is_premium  BOOLEAN      DEFAULT 0
);
CREATE TABLE IF NOT EXISTS products (
    product_id   INT          PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category     VARCHAR(50)  NOT NULL,
    brand        VARCHAR(50)  NOT NULL,
    unit_price   DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    stock_qty    INT           NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
);
CREATE TABLE IF NOT EXISTS orders (
    order_id     INT           PRIMARY KEY,
    customer_id  INT           NOT NULL,
    order_date   DATE          NOT NULL,
    status       VARCHAR(20)   NOT NULL DEFAULT 'Pending'
                 CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
CREATE TABLE IF NOT EXISTS order_items (
    item_id      INT           PRIMARY KEY,
    order_id     INT           NOT NULL,
    product_id   INT           NOT NULL,
    quantity     INT           NOT NULL CHECK (quantity > 0),
    unit_price   DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    discount_pct DECIMAL(5,2)  DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

cur.executescript("""
DELETE FROM order_items; DELETE FROM orders; DELETE FROM products; DELETE FROM customers;
INSERT INTO customers VALUES
(101,'Aarav','Sharma','aarav.s@email.com','Mumbai','Maharashtra','2024-01-15',1),
(102,'Priya','Patel','priya.p@email.com','Ahmedabad','Gujarat','2024-02-20',0),
(103,'Rohan','Gupta','rohan.g@email.com','Delhi','Delhi','2024-03-10',1),
(104,'Sneha','Reddy','sneha.r@email.com','Hyderabad','Telangana','2024-04-05',0),
(105,'Vikram','Singh','vikram.s@email.com','Jaipur','Rajasthan','2024-05-12',1),
(106,'Ananya','Iyer','ananya.i@email.com','Chennai','Tamil Nadu','2024-06-18',0),
(107,'Karan','Mehta','karan.m@email.com','Pune','Maharashtra','2024-07-22',1),
(108,'Divya','Nair','divya.n@email.com','Kochi','Kerala','2024-08-30',0);
INSERT INTO products VALUES
(201,'Wireless Earbuds','Electronics','BoAt',1499.00,250),
(202,'Cotton T-Shirt','Clothing','Levis',799.00,500),
(203,'Smart Watch','Electronics','Noise',2999.00,150),
(204,'Running Shoes','Clothing','Nike',4599.00,120),
(205,'Bluetooth Speaker','Electronics','JBL',3499.00,200),
(206,'Bedsheet Set','Home','Spaces',1299.00,300),
(207,'Laptop Stand','Electronics','AmazonBasics',899.00,180),
(208,'Cushion Covers (Set)','Home','HomeCenter',599.00,400);
INSERT INTO orders VALUES
(1001,101,'2024-08-01','Delivered',4498.00),
(1002,102,'2024-08-03','Delivered',799.00),
(1003,103,'2024-08-05','Shipped',7498.00),
(1004,101,'2024-08-10','Delivered',3499.00),
(1005,104,'2024-08-12','Cancelled',2999.00),
(1006,105,'2024-08-15','Delivered',5898.00),
(1007,106,'2024-08-18','Pending',1299.00),
(1008,103,'2024-08-20','Delivered',899.00),
(1009,107,'2024-08-25','Shipped',6098.00),
(1010,108,'2024-08-28','Delivered',1598.00);
INSERT INTO order_items VALUES
(5001,1001,201,2,1499.00,0),(5002,1001,207,1,899.00,10),
(5003,1002,202,1,799.00,0),(5004,1003,203,1,2999.00,0),
(5005,1003,204,1,4599.00,5),(5006,1004,205,1,3499.00,0),
(5007,1005,203,1,2999.00,0),(5008,1006,201,1,1499.00,10),
(5009,1006,204,1,4599.00,5),(5010,1007,206,1,1299.00,0),
(5011,1008,207,1,899.00,0),(5012,1009,205,1,3499.00,0),
(5013,1009,208,2,599.00,15),(5014,1010,206,1,1299.00,0),
(5015,1010,208,1,599.00,0);
""")
conn.commit()
print('✅ All 4 tables created and data loaded!')
print(f'   customers  : {conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0]} rows')
print(f'   products   : {conn.execute("SELECT COUNT(*) FROM products").fetchone()[0]} rows')
print(f'   orders     : {conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]} rows')
print(f'   order_items: {conn.execute("SELECT COUNT(*) FROM order_items").fetchone()[0]} rows')

✅ All 4 tables created and data loaded!
   customers  : 8 rows
   products   : 8 rows
   orders     : 10 rows
   order_items: 15 rows


---
## 📘 Section A — SQL Basics (Q1–Q6)

In [3]:
# Q1. All columns from customers
print('Q1 — SELECT * FROM customers')
run('SELECT * FROM customers')

Q1 — SELECT * FROM customers


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


In [4]:
# Q2. Only first_name, last_name, city
print('Q2 — SELECT first_name, last_name, city')
run('SELECT first_name, last_name, city FROM customers')

Q2 — SELECT first_name, last_name, city


,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


In [5]:
# Q3. Unique categories
print('Q3 — DISTINCT categories')
run('SELECT DISTINCT category FROM products')

Q3 — DISTINCT categories


,category
0,Electronics
1,Clothing
2,Home


In [6]:
# Q4. Primary Keys of all tables
print('Q4 — Primary Keys')
for table in ['customers','products','orders','order_items']:
    df = pd.read_sql_query(f'PRAGMA table_info({table})', conn)
    pk = df[df['pk'] == 1]['name'].values
    print(f'  {table:15} → PK: {pk}')

Q4 — Primary Keys
  customers       → PK: <StringArray>
['customer_id']
Length: 1, dtype: str
  products        → PK: <StringArray>
['product_id']
Length: 1, dtype: str
  orders          → PK: <StringArray>
['order_id']
Length: 1, dtype: str
  order_items     → PK: <StringArray>
['item_id']
Length: 1, dtype: str


In [7]:
# Q5. UNIQUE constraint test — duplicate email
print('Q5 — Testing UNIQUE constraint on email')
try:
    conn.execute("INSERT INTO customers VALUES (109,'Test','User','aarav.s@email.com','Delhi','Delhi','2024-09-01',0)")
    print('Inserted — no error (unexpected)')
except Exception as e:
    print(f'✅ UNIQUE constraint worked! Error: {e}')

Q5 — Testing UNIQUE constraint on email
✅ UNIQUE constraint worked! Error: UNIQUE constraint failed: customers.email


In [8]:
# Q6. CHECK constraint test — negative price
print('Q6 — Testing CHECK constraint on unit_price')
try:
    conn.execute("INSERT INTO products VALUES (209,'Fake','Electronics','X',-50.00,100)")
    print('Inserted — no error (unexpected)')
except Exception as e:
    print(f'✅ CHECK constraint worked! Error: {e}')

Q6 — Testing CHECK constraint on unit_price
✅ CHECK constraint worked! Error: CHECK constraint failed: unit_price > 0


---
## 📗 Section B — Filtering & Indexes (Q7–Q12)

In [9]:
# Q7. All Delivered orders
print('Q7 — Delivered orders')
run("SELECT * FROM orders WHERE status = 'Delivered'")

Q7 — Delivered orders


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598


In [10]:
# Q8. Electronics with price > 2000
print('Q8 — Electronics with unit_price > 2000')
run("SELECT * FROM products WHERE category = 'Electronics' AND unit_price > 2000")

Q8 — Electronics with unit_price > 2000


,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200


In [11]:
# Q9. Customers from Maharashtra joined in 2024
print('Q9 — Maharashtra customers, joined 2024')
run("SELECT * FROM customers WHERE state = 'Maharashtra' AND join_date LIKE '2024%'")

Q9 — Maharashtra customers, joined 2024


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


In [12]:
# Q10. Orders between Aug 10–25, not cancelled
print('Q10 — Orders Aug 10-25, not cancelled')
run("SELECT * FROM orders WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25' AND status != 'Cancelled'")

Q10 — Orders Aug 10-25, not cancelled


,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499
1,1006,105,2024-08-15,Delivered,5898
2,1007,106,2024-08-18,Pending,1299
3,1008,103,2024-08-20,Delivered,899
4,1009,107,2024-08-25,Shipped,6098


In [13]:
# Q11. Index benefit explanation + sample query
print('Q11 — Index on order_date (idx_orders_date)')
print()
print('Without index: Full table scan — checks every row')
print('With index   : B-tree lookup — jumps directly to matching rows')
print()
print('Query that benefits from idx_orders_date:')
run("SELECT * FROM orders WHERE order_date = '2024-08-15'")

Q11 — Index on order_date (idx_orders_date)

Without index: Full table scan — checks every row
With index   : B-tree lookup — jumps directly to matching rows

Query that benefits from idx_orders_date:


,order_id,customer_id,order_date,status,total_amount
0,1006,105,2024-08-15,Delivered,5898


In [14]:
# Q12. SARGable vs Non-SARGable query
print('Q12 — SARGable query (index-friendly)')
print()
print('❌ Non-SARGable (index NOT used):')
print("   SELECT * FROM customers WHERE YEAR(join_date) = 2024")
print("   Reason: Function on column breaks index usage")
print()
print('✅ SARGable (index IS used):')
print("   SELECT * FROM customers WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31'")
run("SELECT * FROM customers WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31'")

Q12 — SARGable query (index-friendly)

❌ Non-SARGable (index NOT used):
   SELECT * FROM customers WHERE YEAR(join_date) = 2024
   Reason: Function on column breaks index usage

✅ SARGable (index IS used):
   SELECT * FROM customers WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31'


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


---
## 📙 Section C — Aggregations (Q13–Q18)

In [15]:
# Q13. Total orders count
print('Q13 — Total orders')
run('SELECT COUNT(*) AS total_orders FROM orders')

Q13 — Total orders


,total_orders
0,10


In [16]:
# Q14. Revenue from Delivered orders
print('Q14 — Total revenue from Delivered orders')
run("SELECT SUM(total_amount) AS total_revenue FROM orders WHERE status = 'Delivered'")

Q14 — Total revenue from Delivered orders


,total_revenue
0,17191


In [17]:
# Q15. Average price per category
print('Q15 — Avg unit_price per category')
run('SELECT category, ROUND(AVG(unit_price),2) AS avg_price FROM products GROUP BY category')

Q15 — Avg unit_price per category


,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


In [18]:
# Q16. Count and revenue per order status
print('Q16 — Orders count and revenue by status')
run('''
SELECT status,
       COUNT(*) AS order_count,
       ROUND(SUM(total_amount),2) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC
''')

Q16 — Orders count and revenue by status


,status,order_count,total_revenue
0,Delivered,6,17191.0
1,Shipped,2,13596.0
2,Cancelled,1,2999.0
3,Pending,1,1299.0


In [19]:
# Q17. Most expensive and cheapest per category
print('Q17 — MAX and MIN price per category')
run('''
SELECT category,
       MAX(unit_price) AS most_expensive,
       MIN(unit_price) AS cheapest
FROM products
GROUP BY category
''')

Q17 — MAX and MIN price per category


,category,most_expensive,cheapest
0,Clothing,4599,799
1,Electronics,3499,899
2,Home,1299,599


In [20]:
# Q18. Categories with avg price > 2000 (HAVING)
print('Q18 — Categories where AVG price > 2000')
run('''
SELECT category,
       ROUND(AVG(unit_price),2) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000
''')

Q18 — Categories where AVG price > 2000


,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


---
## 📕 Section D — Joins (Q19–Q23)

In [21]:
# Q19. INNER JOIN orders + customers
print('Q19 — INNER JOIN: orders with customer names')
run('''
SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
''')

Q19 — INNER JOIN: orders with customer names


,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498
1,1002,2024-08-03,Priya,Patel,799
2,1003,2024-08-05,Rohan,Gupta,7498
3,1004,2024-08-10,Aarav,Sharma,3499
4,1005,2024-08-12,Sneha,Reddy,2999
5,1006,2024-08-15,Vikram,Singh,5898
6,1007,2024-08-18,Ananya,Iyer,1299
7,1008,2024-08-20,Rohan,Gupta,899
8,1009,2024-08-25,Karan,Mehta,6098
9,1010,2024-08-28,Divya,Nair,1598


In [22]:
# Q20. LEFT JOIN — all customers even without orders
print('Q20 — LEFT JOIN: all customers with their orders')
run('''
SELECT c.customer_id, c.first_name, c.last_name,
       o.order_id, o.order_date, o.status, o.total_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
''')

Q20 — LEFT JOIN: all customers with their orders


,customer_id,first_name,last_name,order_id,order_date,status,total_amount
0,101,Aarav,Sharma,1001,2024-08-01,Delivered,4498
1,101,Aarav,Sharma,1004,2024-08-10,Delivered,3499
2,102,Priya,Patel,1002,2024-08-03,Delivered,799
3,103,Rohan,Gupta,1003,2024-08-05,Shipped,7498
4,103,Rohan,Gupta,1008,2024-08-20,Delivered,899
5,104,Sneha,Reddy,1005,2024-08-12,Cancelled,2999
6,105,Vikram,Singh,1006,2024-08-15,Delivered,5898
7,106,Ananya,Iyer,1007,2024-08-18,Pending,1299
8,107,Karan,Mehta,1009,2024-08-25,Shipped,6098
9,108,Divya,Nair,1010,2024-08-28,Delivered,1598


In [23]:
# Q21. 3-table JOIN: orders + order_items + products
print('Q21 — 3 Table JOIN: order details with product names')
run('''
SELECT o.order_id, p.product_name,
       oi.quantity, oi.unit_price, oi.discount_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p    ON oi.product_id = p.product_id
''')

Q21 — 3 Table JOIN: order details with product names


,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0


In [24]:
# Q22. LEFT vs RIGHT JOIN explanation
print('Q22 — LEFT JOIN vs RIGHT JOIN')
print()
print('LEFT JOIN  → Saare customers dikhte hain, order na ho toh NULL')
print('RIGHT JOIN → Saare orders dikhte hain, customer na ho toh NULL')
print('FULL OUTER → Dono tables ki saari rows — match ho ya na ho')
print()
print('Example — LEFT JOIN (customer 108 has no items in order_items):')
run('''
SELECT c.first_name, c.last_name, o.order_id, o.status
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id
''')

Q22 — LEFT JOIN vs RIGHT JOIN

LEFT JOIN  → Saare customers dikhte hain, order na ho toh NULL
RIGHT JOIN → Saare orders dikhte hain, customer na ho toh NULL
FULL OUTER → Dono tables ki saari rows — match ho ya na ho

Example — LEFT JOIN (customer 108 has no items in order_items):


,first_name,last_name,order_id,status
0,Aarav,Sharma,1001,Delivered
1,Aarav,Sharma,1004,Delivered
2,Priya,Patel,1002,Delivered
3,Rohan,Gupta,1003,Shipped
4,Rohan,Gupta,1008,Delivered
5,Sneha,Reddy,1005,Cancelled
6,Vikram,Singh,1006,Delivered
7,Ananya,Iyer,1007,Pending
8,Karan,Mehta,1009,Shipped
9,Divya,Nair,1010,Delivered


In [25]:
# Q23. Foreign Key test — invalid customer_id
print('Q23 — Foreign Key constraint test')
print()
print('Foreign Key relationships:')
print('  orders.customer_id     → customers.customer_id')
print('  order_items.order_id   → orders.order_id')
print('  order_items.product_id → products.product_id')
print()
conn.execute('PRAGMA foreign_keys = ON')
try:
    conn.execute("INSERT INTO orders VALUES (1011, 999, '2024-09-01', 'Pending', 500.00)")
    conn.commit()
    print('Inserted (FK not enforced in this SQLite mode)')
except Exception as e:
    print(f'✅ FK constraint blocked insert! Error: {e}')

Q23 — Foreign Key constraint test

Foreign Key relationships:
  orders.customer_id     → customers.customer_id
  order_items.order_id   → orders.order_id
  order_items.product_id → products.product_id

Inserted (FK not enforced in this SQLite mode)


---
## 📒 Section E — Advanced: CASE, ACID, Transactions (Q24–Q27)

In [26]:
# Q24. CASE — Price tiers
print('Q24 — Price tier classification using CASE')
run('''
SELECT product_name, unit_price,
    CASE
        WHEN unit_price < 1000            THEN 'Budget'
        WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
        ELSE                                   'Premium'
    END AS price_tier
FROM products
ORDER BY unit_price
''')

Q24 — Price tier classification using CASE


,product_name,unit_price,price_tier
0,Cushion Covers (Set),599,Budget
1,Cotton T-Shirt,799,Budget
2,Laptop Stand,899,Budget
3,Bedsheet Set,1299,Mid-Range
4,Wireless Earbuds,1499,Mid-Range
5,Smart Watch,2999,Mid-Range
6,Bluetooth Speaker,3499,Premium
7,Running Shoes,4599,Premium


In [27]:
# Q25. CASE inside aggregate
print('Q25 — Delivered vs Not Delivered count in one row')
run('''
SELECT
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered,
    SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS not_delivered
FROM orders
''')

Q25 — Delivered vs Not Delivered count in one row


,delivered,not_delivered
0,6,5


In [28]:
# Q26. ACID explanation
print('Q26 — ACID Properties')
print()
print('A — Atomicity   : Transaction ke saare steps hote hain ya koi nahi')
print('                  Example: Bank transfer mein debit hua par credit fail')
print('                  → Poora transaction rollback hoga')
print()
print('C — Consistency : DB hamesha valid state mein rahega')
print('                  Example: Account balance kabhi negative nahi hoga')
print()
print('I — Isolation   : Ek transaction doosre ko affect nahi karta jab tak commit na ho')
print('                  Example: 2 log ek saath same seat book nahi kar sakte')
print()
print('D — Durability  : Commit ke baad data permanently save rehta hai')
print('                  Example: Payment complete hone ke baad system crash bhi ho')
print('                  → Data lost nahi hoga')

Q26 — ACID Properties

A — Atomicity   : Transaction ke saare steps hote hain ya koi nahi
                  Example: Bank transfer mein debit hua par credit fail
                  → Poora transaction rollback hoga

C — Consistency : DB hamesha valid state mein rahega
                  Example: Account balance kabhi negative nahi hoga

I — Isolation   : Ek transaction doosre ko affect nahi karta jab tak commit na ho
                  Example: 2 log ek saath same seat book nahi kar sakte

D — Durability  : Commit ke baad data permanently save rehta hai
                  Example: Payment complete hone ke baad system crash bhi ho
                  → Data lost nahi hoga


In [29]:
# Q27. Full Transaction — BEGIN / COMMIT / ROLLBACK
print('Q27 — Transaction: New order + items + stock update')
print()
try:
    conn.execute('BEGIN')

    # Step 1: Insert new order
    conn.execute("INSERT INTO orders VALUES (1011, 102, '2024-09-01', 'Pending', 1598.00)")
    print('  ✅ Step 1: New order 1011 inserted')

    # Step 2: Insert order items
    conn.execute("INSERT INTO order_items VALUES (5016, 1011, 206, 1, 1299.00, 0)")
    conn.execute("INSERT INTO order_items VALUES (5017, 1011, 208, 1, 599.00, 0)")
    print('  ✅ Step 2: 2 order items inserted')

    # Step 3: Update stock
    conn.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206")
    conn.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208")
    print('  ✅ Step 3: Stock updated for products 206 and 208')

    conn.execute('COMMIT')
    print()
    print('  ✅ COMMIT successful — all steps saved!')

except Exception as e:
    conn.execute('ROLLBACK')
    print(f'  ❌ Error: {e}')
    print('  ↩️  ROLLBACK — all steps undone!')

print()
print('Verify new order:')
run("SELECT * FROM orders WHERE order_id = 1011")

Q27 — Transaction: New order + items + stock update

  ❌ Error: UNIQUE constraint failed: orders.order_id
  ↩️  ROLLBACK — all steps undone!

Verify new order:


,order_id,customer_id,order_date,status,total_amount
0,1011,999,2024-09-01,Pending,500


In [30]:
# Updated stock verification
print('Verify stock updated:')
run("SELECT product_id, product_name, stock_qty FROM products WHERE product_id IN (206, 208)")

Verify stock updated:


,product_id,product_name,stock_qty
0,206,Bedsheet Set,300
1,208,Cushion Covers (Set),400


---
## 🔍 Key Business Insights

| Insight | Finding |
|---|---|
| Top Revenue Status | Delivered orders generate maximum revenue |
| Most Expensive Category | Electronics has highest avg price ₹2,224 |
| Most Active Customer | Aarav Sharma (101) — 2 orders |
| Premium Members | 4 out of 8 customers are premium |
| Highest Single Order | ₹7,498 — Rohan Gupta (Order 1003) |
| Loss Risk | Cancelled orders = ₹2,999 lost revenue |

In [31]:
conn.close()
print('✅ Analysis complete! Connection closed.')
print('📁 Files in this project:')
print('   shopease_analysis.ipynb  — This notebook')
print('   shopease.db              — SQLite database')
print('   schema.sql               — CREATE TABLE statements')
print('   insert_data.sql          — Sample data')
print('   section_a.sql to e.sql   — Individual SQL files')
print('   README.md                — Project documentation')

✅ Analysis complete! Connection closed.
📁 Files in this project:
   shopease_analysis.ipynb  — This notebook
   shopease.db              — SQLite database
   schema.sql               — CREATE TABLE statements
   insert_data.sql          — Sample data
   section_a.sql to e.sql   — Individual SQL files
   README.md                — Project documentation
